## Notebook ini berisi proses scrapping data yang diperlukan untuk projek Tomato Leafguard

### Data yang diperlukan antara lain:


1.   Gambar daun dengan background (color) https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset?select=color
2.   Gambar daun tanpa background (segmented) https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset?select=segmented
3.   Gambar acak sebagai kelas tak terdefinisi yang berasal dari 3 sumber data yang terdapat di kaggle, antara lain:

      *   https://www.kaggle.com/datasets/puneet6060/intel-image-classification
      *   https://www.kaggle.com/datasets/prasunroy/natural-images
      *   https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset





# 1. Import Libary dan Konfigurasi Direktori Colab

In [2]:
import os
import json
import zipfile
import random
import pandas as pd
import shutil
from google.colab import files
from kaggle.api.kaggle_api_extended import KaggleApi

jalankan dua kali

In [3]:
# KONFIGURASI GLOBAL DIRECTORY
COLOR_DIR = "/content/color"
SEGMENTED_DIR = "/content/segmented"
UNIDEN_DIR = "/content/Tak_Terdefinisi_Master"
TOMATO_ZIP = "plantvillage-dataset.zip"

# 2. Autentifikasi Kaggle API

untuk melakukan ini diperlukan file kaggle.json yang bisa didapat dari Kaggle API

In [4]:
# AUTENTIKASI KAGGLE API
if os.path.exists('/content/kaggle.json'):
    with open('/content/kaggle.json') as f:
        config = json.load(f)
        os.environ['KAGGLE_USERNAME'] = config['username']
        os.environ['KAGGLE_KEY'] = config['key']
    print("Kredensial Kaggle siap digunakan.")
else:
    print("Warning: File kaggle.json belum diunggah ke direktori /content/ Colab!")

Kredensial Kaggle siap digunakan.


# 3. Ekstrak data gambar untuk kelas tak terdefinisi

In [5]:
# FUNGSI EKSTRAKSI GAMBAR ACAK UNTUK KELAS TAK TERDEFINISI
def extract_random_images_from_zip(zip_path, target_dir, num_images, prefix):
    print(f"Mengambil {num_images} gambar acak dari {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Kumpulkan semua file gambar dari berbagai sub-folder
        image_files = [
            m for m in zf.infolist()
            if not m.is_dir() and m.filename.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]

        # Pilih gambar secara acak
        selected_files = random.sample(image_files, min(num_images, len(image_files)))

        for i, member in enumerate(selected_files):
            ext = os.path.splitext(member.filename)[1]
            new_filename = f"{prefix}_{i}{ext}"
            target_path = os.path.join(target_dir, new_filename)

            # Tulis file yang dipilih ke folder penampungan
            with zf.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())
    print(f"Selesai mengekstrak {len(selected_files)} gambar dengan prefix '{prefix}'.")

# 4. Mengunduh Dataset tiap kelas

In [6]:
# PROSES UNDUHAN DATASET
try:
    print("Menghubungkan ke Kaggle API...")
    api = KaggleApi()
    api.authenticate()

    os.makedirs(COLOR_DIR, exist_ok=True)
    os.makedirs(SEGMENTED_DIR, exist_ok=True)
    os.makedirs(UNIDEN_DIR, exist_ok=True)

    # -------------------------------------------------------------
    # TAHAP 1: MENGAMBIL DATA DAUN TOMAT (COLOR & SEGMENTED)
    # -------------------------------------------------------------
    print("\n[1/5] Mengunduh dataset PlantVillage (Hanya Tomat)...")
    api.dataset_download_files("abdallahalidev/plantvillage-dataset", path='.', unzip=False)

    with zipfile.ZipFile(TOMATO_ZIP, 'r') as zf:
        for member in zf.infolist():
            if "Tomato" in member.filename:
                # Memproses data Color
                if "plantvillage dataset/color/" in member.filename:
                    filename_rel = member.filename.replace("plantvillage dataset/color/", "")
                    if filename_rel:
                        target_path = os.path.join(COLOR_DIR, filename_rel)
                        if member.is_dir():
                            os.makedirs(target_path, exist_ok=True)
                        else:
                            os.makedirs(os.path.dirname(target_path), exist_ok=True)
                            with zf.open(member) as source, open(target_path, "wb") as target:
                                target.write(source.read())

                # Memproses data Segmented
                elif "plantvillage dataset/segmented/" in member.filename:
                    filename_rel = member.filename.replace("plantvillage dataset/segmented/", "")
                    if filename_rel:
                        target_path = os.path.join(SEGMENTED_DIR, filename_rel)
                        if member.is_dir():
                            os.makedirs(target_path, exist_ok=True)
                        else:
                            os.makedirs(os.path.dirname(target_path), exist_ok=True)
                            with zf.open(member) as source, open(target_path, "wb") as target:
                                target.write(source.read())

    os.remove(TOMATO_ZIP)
    print("Data Color dan Segmented daun tomat berhasil dipilah.")

    # -------------------------------------------------------------
    # TAHAP 2 - 4: MENGAMBIL DATA TAK TERDEFINISI
    # -------------------------------------------------------------
    print("\n[2/5] Mengunduh dataset Intel Image Classification...")
    api.dataset_download_files("puneet6060/intel-image-classification", path='.', unzip=False)
    extract_random_images_from_zip("intel-image-classification.zip", UNIDEN_DIR, 333, "intel")
    os.remove("intel-image-classification.zip")

    print("\n[3/5] Mengunduh dataset Natural Images...")
    api.dataset_download_files("prasunroy/natural-images", path='.', unzip=False)
    extract_random_images_from_zip("natural-images.zip", UNIDEN_DIR, 333, "natural")
    os.remove("natural-images.zip")

    print("\n[4/5] Mengunduh dataset COCO 2017 (val2017)...")
    try:
        api.dataset_download_file("awsaf49/coco-2017-dataset", "val2017.zip", path='.')
        extract_random_images_from_zip("val2017.zip", UNIDEN_DIR, 334, "coco")
        os.remove("val2017.zip")
    except Exception as e:
        print("Mencoba mengunduh keseluruhan COCO...")
        api.dataset_download_files("awsaf49/coco-2017-dataset", path='.', unzip=False)
        extract_random_images_from_zip("coco-2017-dataset.zip", UNIDEN_DIR, 334, "coco")
        os.remove("coco-2017-dataset.zip")

    # -------------------------------------------------------------
    # TAHAP 5: MENYALIN KELAS TAK TERDEFINISI KE KEDUA DATASET
    # -------------------------------------------------------------
    print("\n[5/5] Menduplikasi kelas 'Tak_Terdefinisi' ke dataset Color dan Segmented...")
    shutil.copytree(UNIDEN_DIR, os.path.join(COLOR_DIR, "Tak_Terdefinisi"), dirs_exist_ok=True)
    shutil.copytree(UNIDEN_DIR, os.path.join(SEGMENTED_DIR, "Tak_Terdefinisi"), dirs_exist_ok=True)

    print(f"\nSEMUA PROSES SELESAI!")
    print(f"Dataset Color siap di: {COLOR_DIR}")
    print(f"Dataset Segmented siap di: {SEGMENTED_DIR}")

except Exception as e:
    print(f"Terjadi gangguan saat eksekusi: {e}")

Menghubungkan ke Kaggle API...

[1/5] Mengunduh dataset PlantVillage (Hanya Tomat)...
Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
Data Color dan Segmented daun tomat berhasil dipilah.

[2/5] Mengunduh dataset Intel Image Classification...
Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
Mengambil 333 gambar acak dari intel-image-classification.zip...
Selesai mengekstrak 333 gambar dengan prefix 'intel'.

[3/5] Mengunduh dataset Natural Images...
Dataset URL: https://www.kaggle.com/datasets/prasunroy/natural-images
Mengambil 333 gambar acak dari natural-images.zip...
Selesai mengekstrak 333 gambar dengan prefix 'natural'.

[4/5] Mengunduh dataset COCO 2017 (val2017)...
Dataset URL: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
Mencoba mengunduh keseluruhan COCO...
Dataset URL: https://www.kaggle.com/datasets/awsaf49/coco-2017-dataset
Mengambil 334 gambar acak dari coco-2017-dataset.zip...
Selesai mengekst

# 5. Validasi Dataset

In [7]:
# --- VALIDASI HASIL AKHIR ---
print("\nMEMERIKSA HASIL DISTRIBUSI DATA...")

def cek_distribusi(path_dir):
    ringkasan = []
    if os.path.exists(path_dir):
        for folder in sorted(os.listdir(path_dir)):
            folder_full_path = os.path.join(path_dir, folder)
            if os.path.isdir(folder_full_path) and not folder.startswith('.'):
                ringkasan.append({'Nama Kategori/Penyakit': folder, 'Jumlah File': len(os.listdir(folder_full_path))})
    return pd.DataFrame(ringkasan)

df_color = cek_distribusi(COLOR_DIR)
df_segmented = cek_distribusi(SEGMENTED_DIR)

print("\nDATA TOMAT - COLOR:")
if not df_color.empty:
    print(f"Total: {df_color['Jumlah File'].sum()} Gambar")
    print(df_color.to_string(index=False))

print("\nDATA TOMAT - SEGMENTED:")
if not df_segmented.empty:
    print(f"Total: {df_segmented['Jumlah File'].sum()} Gambar")
    print(df_segmented.to_string(index=False))


MEMERIKSA HASIL DISTRIBUSI DATA...

DATA TOMAT - COLOR:
Total: 19160 Gambar
                       Nama Kategori/Penyakit  Jumlah File
                              Tak_Terdefinisi         1000
                      Tomato___Bacterial_spot         2127
                        Tomato___Early_blight         1000
                         Tomato___Late_blight         1909
                           Tomato___Leaf_Mold          952
                  Tomato___Septoria_leaf_spot         1771
Tomato___Spider_mites Two-spotted_spider_mite         1676
                         Tomato___Target_Spot         1404
       Tomato___Tomato_Yellow_Leaf_Curl_Virus         5357
                 Tomato___Tomato_mosaic_virus          373
                             Tomato___healthy         1591

DATA TOMAT - SEGMENTED:
Total: 19160 Gambar
                       Nama Kategori/Penyakit  Jumlah File
                              Tak_Terdefinisi         1000
                      Tomato___Bacterial_spot       

In [8]:
zip_filename = "segmented.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', SEGMENTED_DIR)

'/content/segmented.zip'

**dataset sudah tersimpan lalu dikonversi dalam bentuk zip agar bisa diunduh, setelahnya dataset akan diunggah ke data dictionary agar mudah diakses oleh tim Data Scientist dan AI Engineer**

github: https://github.com/happy-ending-forever/capstone-tomato-leafguard